# Getting Started with the FunGen-xQTL Protocol

**A reproducible, end-to-end computational protocol for molecular quantitative trait loci (xQTL) analysis — bulk, single-cell, and multi-omic.**

The FunGen-xQTL protocol brings together data preprocessing, QTL discovery, fine-mapping, multivariate and colocalization analyses, and integration with GWAS into a single reproducible workflow. Every step is implemented as a [Script of Scripts (SoS)](https://vatlab.github.io/sos-docs/) workflow that runs the same way on a laptop, a compute cluster, or the cloud.

> **New here?** This page is a guided on-ramp. In about an hour you can install the environment, clone the repo, download a small demo dataset, and run your first pipeline.

---

## At a Glance

| | |
|---|---|
| **Code repository** | [StatFunGen/xqtl-protocol](https://github.com/StatFunGen/xqtl-protocol) |
| **Project website** | [statfungen.github.io/xqtl-protocol](https://statfungen.github.io/xqtl-protocol/) |
| **Workflow engine** | [SoS (Script of Scripts)](https://vatlab.github.io/sos-docs/) |
| **Package manager** | [pixi](https://pixi.sh/) via [StatFunGen/pixi-setup](https://github.com/StatFunGen/pixi-setup) |
| **Lab environment guide** | [wanggroup.org — Software Setup](https://wanggroup.org/hpc/docs/software-setup-conda) |
| **Supported OS** | Linux, macOS 11+. Windows via [WSL](https://learn.microsoft.com/windows/wsl/install) |
| **HPC schedulers** | SLURM, LSF, SGE, PBS/Torque |

---

## How to Use This Page

1. **Check prerequisites** — make sure you're on a supported OS and have enough disk/memory.
2. **Install the environment** — run the `pixi-setup.sh` installer, then add SoS.
3. **Get the code and data** — clone the protocol repo and download the demo dataset.
4. **Run your first workflow** — execute a small example to confirm everything works.
5. **Go further** — follow the links at the bottom of this page to dive into the pipelines that matter for your project.

Each section below is self-contained; if something is already installed you can skip ahead.

---

## 1. Prerequisites

Before you start, confirm the following:

| Requirement | Minimum | Recommended |
|---|---|---|
| **Operating system** | Linux or macOS 11+ (Windows users: install [WSL2](https://learn.microsoft.com/windows/wsl/install) first) | Ubuntu 22.04+ or macOS 13+ |
| **Shell** | Bash or Zsh | Zsh on macOS, Bash on Linux/HPC |
| **Memory** | 16 GB | 50 GB+ for the pixi installer on HPC |
| **Disk space** | 10 GB (minimal install) | 50 GB+ (full bioinformatics stack + demo data) |
| **Network access** | GitHub, conda-forge, synapse.org | Same |
| **Git** | Any recent version | 2.30+ |

> **HPC tip:** Request an interactive compute node with at least 50 GB of memory before running the installer. Login nodes often kill large installs.
>
> ```bash
> # SLURM
> srun --mem=50G --pty bash
>
> # LSF
> bsub -Is -M 50000 -n 4 bash
> ```

---

## 2. Install the Computing Environment

We manage the entire software stack with [**pixi**](https://pixi.sh/), a fast, reproducible package manager for conda channels. The [StatFunGen/pixi-setup](https://github.com/StatFunGen/pixi-setup) installer wires everything up for you — Python, R, JupyterLab, bioinformatics tools, and more — and is the **same installer the Wang Lab uses**. The canonical walkthrough lives at [wanggroup.org/hpc/docs/software-setup-conda](https://wanggroup.org/hpc/docs/software-setup-conda); the steps below mirror it.

### 2.1 (Optional) Purge old conda/mamba installs

If you already have conflicting `conda`, `mamba`, `micromamba`, or a prior `pixi` on the machine, start from a clean slate:

```bash
rm -rf ~/.mamba ~/.conda ~/.anaconda ~/.pixi ~/.jupyter \
       ~/micromamba ~/.mambarc ~/.local/share/jupyter/
```

Skip this step if you are happy with your current setup or sharing the machine with others.

### 2.2 Run the pixi-setup installer

From an interactive shell (or a compute node on HPC):

```bash
curl -fsSL https://raw.githubusercontent.com/StatFunGen/pixi-setup/refs/heads/main/pixi-setup.sh \
     -o pixi-setup.sh
bash pixi-setup.sh
```

You'll be prompted for two things.

**Installation path** — where pixi stores environments and the package cache.

| Setting | When to use |
|---|---|
| `$HOME/.pixi` (default) | Laptops and workstations with plenty of `$HOME` space |
| `/lab/$USER/.pixi` or scratch | HPC systems with small `$HOME` quotas (recommended) |

**Installation type** — pick based on what you plan to do.

| Type | Size | Files | Includes |
|---|---|---|---|
| **1. minimal** | ~5 GB | ~100k | CLI tools, Python data-science stack, JupyterLab, base R (tidyverse, devtools, IRkernel, languageserver, …) |
| **2. full** | ~35 GB | ~350k | Everything in minimal **plus** samtools, bcftools, plink/plink2, GATK4, STAR, RSEM, FastQC, bedtools, VEP, Seurat, tensorQTL, and Bioconductor packages |

> **Rule of thumb:** pick **minimal** for xQTL runs where you pass in pre-processed data; pick **full** if you'll also do upstream QC, alignment, or single-cell preprocessing.

After the installer finishes:

```bash
source ~/.bashrc      # or ~/.zshrc on macOS
pixi --version        # should print a version number
```

### 2.3 Add SoS on top of pixi (wanggroup.org approach)

Pixi gives you Python, R, and JupyterLab, but the xQTL protocol is written as **SoS workflows**, so we add the SoS suite as pixi global packages. This is exactly the Wang Lab convention from [wanggroup.org](https://wanggroup.org/orientation/jupyter-setup.html) — SoS lives next to pixi's `python` environment and is available from any shell.

```bash
pixi global install \
    --environment python \
    -c conda-forge \
    sos sos-pbs sos-notebook jupyterlab-sos \
    sos-bash sos-python sos-r

# Register the SoS Jupyter kernel
pixi run -e python python -m sos_notebook.install
```

Verify the install:

```bash
sos --version               # SoS workflow CLI
jupyter kernelspec list     # should list 'sos' among the kernels
```

> **Why separate from the pixi installer?** The minimal and full pixi-setup bundles intentionally ship a general-purpose Python + R stack. Adding SoS as its own step keeps the base install portable and makes it easy to upgrade SoS independently.

### 2.4 Install Additional Software (optional)

Once pixi is configured, installing more tools is a one-liner. Consult [anaconda.org](https://anaconda.org/search) for exact package names.

```bash
# Bioinformatics CLI tools
pixi global install -c bioconda samtools bcftools plink2

# R package (into the r-base environment)
pixi global install -c conda-forge --environment r-base r-pacman

# Python package (into the python environment)
pixi global install -c conda-forge --environment python seaborn

# Update all packages in an environment
pixi global update r-base
pixi global update python
```

### 2.5 (Optional) Collaboration-friendly permissions

If you share a lab directory, make new files group-writable by default. Add to `~/.bashrc`:

```bash
umask 002
```

---

## 3. Get the Code

Clone the protocol repository — this is where every pipeline, config, and example lives.

```bash
git clone https://github.com/StatFunGen/xqtl-protocol.git
cd xqtl-protocol
```

### Repository layout

| Path | What it is |
|---|---|
| `code/` | Notebook-based documentation (this page lives here) |
| `pipeline/` | SoS workflow entry points (symlinks into `code/`) — this is what you run |
| `website/` | JupyterBook sources for [statfungen.github.io/xqtl-protocol](https://statfungen.github.io/xqtl-protocol/) |
| `data/` | Small example inputs and configuration templates |
| `container/` | Legacy Singularity/Docker recipes (kept for reference) |

### The pipelines you'll use most

| Entry point | Purpose |
|---|---|
| `pipeline/1_xqtl_association.ipynb` | End-to-end xQTL association pipeline |
| `pipeline/TensorQTL.ipynb` | TensorQTL cis/trans mapping |
| `pipeline/1_phenotype_preprocessing.ipynb` | Phenotype QC, normalization, covariate correction |
| `pipeline/2_genotype_preprocessing.ipynb` | Genotype QC, imputation, reference alignment |
| `pipeline/4_covariates_preprocessing.ipynb` | PEER / hidden-factor covariate generation |
| `pipeline/eQTL_analysis_commands.ipynb` | Copy-paste command reference for eQTL runs |
| `pipeline/Job_Example.ipynb` | Template for submitting pipelines to an HPC scheduler |

Browse the full index on the [pipelines page of the website](https://statfungen.github.io/xqtl-protocol/).

---

## 4. Run Your First Workflow

Once pixi + SoS are installed and the repo is cloned, confirm SoS can see the workflows:

```bash
cd xqtl-protocol
sos run pipeline/1_xqtl_association.ipynb -h
```

You should see a list of available workflow steps and options. If you do, you're ready to launch real runs.

A minimal cis-QTL smoke test (requires demo data — see next section):

```bash
sos run pipeline/TensorQTL.ipynb cis \
    --genotype-file data/example/genotype.bed \
    --phenotype-file data/example/phenotype.bed.gz \
    --covariate-file data/example/covariates.tsv \
    --cwd output/demo_tensorqtl
```

> **Tip:** Every pipeline supports `-h` and `--help`. SoS also prints the exact shell commands it runs, which is handy for debugging and for learning what the pipeline does under the hood.

---

## 5. Where to Go Next

**Explore the protocol**

- [Full documentation site](https://statfungen.github.io/xqtl-protocol/) — browsable pipeline reference with examples
- [Pipeline index](https://statfungen.github.io/xqtl-protocol/pipeline/) — each step with inputs, outputs, and parameters
- [HPC quick start (Wang Lab)](https://wanggroup.org/hpc/docs/quick-start/) — how to request nodes and submit jobs
- [SoS documentation](https://vatlab.github.io/sos-docs/) — workflow engine reference and tutorials

**Get help**

- [Ask a question (Wang Lab guide)](https://wanggroup.org/orientation/questions.html) — how to file a good issue
- [GitHub issues](https://github.com/StatFunGen/xqtl-protocol/issues) — bugs, feature requests, protocol questions
- [pixi-setup issues](https://github.com/StatFunGen/pixi-setup/issues) — environment / install problems

**Contribute**

- Fork [StatFunGen/xqtl-protocol](https://github.com/StatFunGen/xqtl-protocol), make changes on a feature branch, open a PR
- Follow the [reproducible research guide](https://wanggroup.org/orientation/reproducible-research.html) for notebook and commit conventions

---


## 6. Analysis Overview

The FunGen-xQTL protocol is modular. Each numbered pipeline is a self-contained SoS notebook that can run independently or as part of the full xQTL workflow. At a high level, the flow is:

**Preprocess inputs → Discover QTLs → Fine-map & integrate → Report & share**

| Stage | Pipelines | What happens |
|---|---|---|
| **1. Phenotype preprocessing** | `1_phenotype_preprocessing.ipynb` | QC, normalization, batch correction for bulk RNA-seq, proteomics, methylation, or single-cell pseudo-bulk |
| **2. Genotype preprocessing** | `2_genotype_preprocessing.ipynb` | Variant-level QC, imputation, ancestry alignment, PCA |
| **3. Covariate preprocessing** | `4_covariates_preprocessing.ipynb` | Known + hidden covariates (PEER, surrogate variables) |
| **4. QTL discovery** | `TensorQTL.ipynb`, `APEX.ipynb`, `1_xqtl_association.ipynb` | Cis/trans scans, interaction QTLs, trans-eQTL screens |
| **5. Fine-mapping & multivariate** | `SuSiE.ipynb`, `mvSuSiE.ipynb`, `fSuSiE.ipynb` | Credible sets across contexts and tissues |
| **6. Colocalization & integration** | `coloc.ipynb`, `cTWAS.ipynb`, `GWAS_integration.ipynb` | Link xQTLs to GWAS and causal gene nomination |

All pipelines share a common config layout, so once you've learned one you can read the others quickly.


## 7. Downloading Example Data

A small demo dataset lives on [Synapse](https://www.synapse.org/#!Synapse:syn36416559). You'll need a free Synapse account, then install the client (already available via pixi):

```bash
# One-time install (uses pixi's python env)
pixi global install -c conda-forge --environment python synapseclient

# Log in (prompts for your Synapse PAT)
synapse login -p

# Download the demo bundle (~2 GB)
synapse get -r syn36416559 --downloadLocation data/example/
```

For the full ROSMAP / MSBB production data (controlled access), request access through [AD Knowledge Portal](https://adknowledgeportal.synapse.org/) first.


## 8. Troubleshooting

A handful of issues come up often. If one of these matches, try the fix before opening an issue.

**`pixi: command not found` after install**

```bash
source ~/.bashrc      # Linux / HPC
source ~/.zshrc       # macOS
# or open a fresh terminal
```

**Installer gets killed on HPC**
Request more memory (≥ 50 GB) and run on a compute node, not the login node:

```bash
srun --mem=50G --pty bash
bash pixi-setup.sh
```

**`sos: command not found`**
SoS was not installed on top of pixi. Re-run step 2.3.

**R library conflicts**
Conda-forge R packages do not mix well with `install.packages()` builds. Prefer `pixi global install --environment r-base r-<pkg>`. If you must use CRAN, stick to pure-R packages.

**`ModuleNotFoundError` during a pipeline**
Install the missing package into pixi's `python` env:

```bash
pixi global install -c conda-forge --environment python <package>
```

**GitHub is unreachable from your network**
Use the Gitee mirror documented at [wanggroup.org/hpc/docs/software-setup-conda](https://wanggroup.org/hpc/docs/software-setup-conda) (see "Users in China").

**File permissions on shared directories**
Add `umask 002` to your `~/.bashrc` so new files are group-writable.

---

## 9. Running on HPC

The protocol plays nicely with SoS's built-in task queue support for **SLURM**, **LSF**, **SGE**, and **PBS/Torque**. See `pipeline/Job_Example.ipynb` for a template and the [SoS task queue docs](https://vatlab.github.io/sos-docs/doc/user_guide/task_statement.html) for configuration.

A typical SLURM submission looks like:

```bash
sos run pipeline/TensorQTL.ipynb cis \
    --genotype-file ... --phenotype-file ... --covariate-file ... \
    --cwd output/run01 \
    -q slurm -c config/hpc_slurm.yml \
    -J 20            # up to 20 concurrent jobs
```

The `config/hpc_slurm.yml` file controls partitions, walltime, and memory per task. Start from the template in the repo and adapt it to your cluster.

---

## 10. Citing the Protocol

If you use the FunGen-xQTL protocol in a publication, please cite:

> Cao *et al.*, *A computational protocol for molecular QTL analysis integrating GWAS.* See [CITATION.md](https://github.com/StatFunGen/xqtl-protocol/blob/main/CITATION.md) in the repository for the current preferred citation and BibTeX entry.

And drop us a line — we love hearing how the protocol is being used.
